# 05. Build Stacked Event-Study Dataset

This notebook constructs the stacked event-study dataset for LP-DiD estimation.
Following Cengiz et al. (2019) and Baker et al. (2022), we create cohort-specific
datasets that avoid TWFE contamination from staggered adoption.

**Inputs:**
- `data/03_clean/panel_with_shocks.parquet`

**Outputs:**
- `data/03_clean/stacked_events_pos.parquet`
- `data/03_clean/stacked_events_neg.parquet`
- `data/03_clean/stacked_events_combined.parquet`

**Key design choices:**
1. Each event (cohort) gets its own subsample
2. Clean controls: never-treated or not-yet-treated
3. Event-time indexing relative to treatment
4. Drop overlapping event windows

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Project paths
PROJECT_ROOT = Path(__file__).parent.parent if '__file__' in dir() else Path.cwd().parent
DATA_CLEAN = PROJECT_ROOT / 'data/03_clean'
OUTPUT_LOGS = PROJECT_ROOT / 'output/logs'

print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/gabrielsaco/Documents/GitHub/economic-freedom


## 1. Load Panel with Shocks

In [2]:
panel_path = DATA_CLEAN / 'panel_with_shocks.parquet'
assert panel_path.exists(), f"Panel data not found: {panel_path}"

df = pd.read_parquet(panel_path)
print(f"Panel: {len(df)} observations, {df['iso3c'].nunique()} countries")
print(f"Years: {sorted(df['year'].unique())}")

Panel: 1815 observations, 165 countries
Years: [np.int64(1970), np.int64(1975), np.int64(1980), np.int64(1985), np.int64(1990), np.int64(1995), np.int64(2000), np.int64(2005), np.int64(2010), np.int64(2015), np.int64(2020)]


## 2. Event Study Configuration

In [3]:
# Event study parameters
EVENT_CONFIG = {
    # Number of pre-event periods (K in paper)
    # Extended from 2 to 4 to use more of the 11 quinquennial periods
    'pre_periods': 4,  # -4, -3, -2, -1 relative to event (20 years of pre-data)
    
    # Number of post-event periods (L in paper)
    # Extended from 4 to 6 to capture longer-run effects
    'post_periods': 6,  # 0, 1, 2, 3, 4, 5, 6 relative to event (30 years of post-data)
    
    # Event variables to use (cooldown-enforced)
    'pos_event_col': 'reform_agg_pos_cd',
    # Note: Use basic shocks (S-) for negative reforms due to sparse sustained reforms
    'neg_event_col': 'shock_agg_neg_cd',  # Alternative: df might have reform_agg_neg_cd_alt
    
    # Control specification
    'control_type': 'not_yet_treated',  # or 'never_treated'
    
    # Minimum observations per event
    'min_obs_per_event': 5,
}

# Derived parameters
K = EVENT_CONFIG['pre_periods']
L = EVENT_CONFIG['post_periods']
window_size = K + L + 1  # Total event window size
YEARS = sorted(df['year'].unique())
QUINQUENNIAL_STEP = 5

print("Event study configuration:")
for key, value in EVENT_CONFIG.items():
    print(f"  {key}: {value}")
print(f"\nEvent window: {-K} to +{L} ({window_size} periods)")

Event study configuration:
  pre_periods: 2
  post_periods: 4
  pos_event_col: reform_agg_pos_cd
  neg_event_col: shock_agg_neg_cd
  control_type: not_yet_treated
  min_obs_per_event: 5

Event window: -2 to +4 (7 periods)


## 3. Identify Event Cohorts

In [4]:
def identify_events(df, event_col):
    """
    Identify all events and their timing.
    
    Returns DataFrame with (iso3c, event_year, event_type)
    """
    events = df[df[event_col] == 1][['iso3c', 'year']].copy()
    events = events.rename(columns={'year': 'event_year'})
    events['event_type'] = event_col
    return events

# Identify positive and negative events
events_pos = identify_events(df, EVENT_CONFIG['pos_event_col'])
events_neg = identify_events(df, EVENT_CONFIG['neg_event_col'])

print(f"Positive reform events: {len(events_pos)}")
print(f"Negative reform events: {len(events_neg)}")

print("\nPositive events by year:")
print(events_pos.groupby('event_year').size())

print("\nNegative events by year:")
print(events_neg.groupby('event_year').size())

Positive reform events: 48
Negative reform events: 20

Positive events by year:
event_year
1980     2
1985     3
1990    13
1995    13
2000     8
2005     8
2010     1
dtype: int64

Negative events by year:
event_year
1975    7
1980    3
1985    1
1990    1
2000    1
2005    2
2010    3
2020    2
dtype: int64


## 4. Build Stacked Dataset for One Event Type

In [5]:
def build_stacked_dataset(df, events_df, K=2, L=4, control_type='not_yet_treated'):
    """
    Build stacked event-study dataset.
    
    For each event cohort (event_year, event_country), we create a subsample
    containing the treated unit and appropriate controls, indexed by event time.
    
    Parameters
    ----------
    df : DataFrame
        Full panel data
    events_df : DataFrame
        Events with columns ['iso3c', 'event_year']
    K : int
        Number of pre-periods
    L : int
        Number of post-periods
    control_type : str
        'never_treated' or 'not_yet_treated'
        
    Returns
    -------
    DataFrame in stacked format with cohort identifiers
    """
    stacked_dfs = []
    
    # Get all treated countries
    treated_countries = set(events_df['iso3c'].unique())
    never_treated = set(df['iso3c'].unique()) - treated_countries
    
    # Get all event years
    all_event_years = sorted(events_df['event_year'].unique())
    
    for idx, (_, event) in enumerate(events_df.iterrows()):
        iso = event['iso3c']
        event_year = event['event_year']
        
        # Define event window
        window_start = event_year - K * QUINQUENNIAL_STEP
        window_end = event_year + L * QUINQUENNIAL_STEP
        
        # Create cohort identifier
        cohort_id = f"{iso}_{event_year}"
        
        # Select treated unit
        treated_data = df[
            (df['iso3c'] == iso) & 
            (df['year'] >= window_start) & 
            (df['year'] <= window_end)
        ].copy()
        treated_data['treated'] = 1
        treated_data['cohort_id'] = cohort_id
        treated_data['cohort_year'] = event_year
        
        # Define post-treatment indicator
        treated_data['post'] = (treated_data['year'] >= event_year).astype(int)
        
        # Event time relative to treatment
        treated_data['event_time'] = (treated_data['year'] - event_year) // QUINQUENNIAL_STEP
        
        # Interaction for DiD
        treated_data['treat_post'] = treated_data['treated'] * treated_data['post']
        
        # Select controls
        if control_type == 'never_treated':
            control_countries = never_treated
        else:  # not_yet_treated
            # Countries that haven't been treated by this event year
            later_treated = events_df[events_df['event_year'] > event_year]['iso3c'].unique()
            control_countries = never_treated | set(later_treated)
        
        if len(control_countries) == 0:
            continue
            
        control_data = df[
            (df['iso3c'].isin(control_countries)) & 
            (df['year'] >= window_start) & 
            (df['year'] <= window_end)
        ].copy()
        control_data['treated'] = 0
        control_data['cohort_id'] = cohort_id
        control_data['cohort_year'] = event_year
        control_data['post'] = (control_data['year'] >= event_year).astype(int)
        control_data['event_time'] = (control_data['year'] - event_year) // QUINQUENNIAL_STEP
        control_data['treat_post'] = 0
        
        # Combine
        cohort_data = pd.concat([treated_data, control_data], ignore_index=True)
        stacked_dfs.append(cohort_data)
    
    if len(stacked_dfs) == 0:
        return pd.DataFrame()
    
    # Stack all cohorts
    stacked = pd.concat(stacked_dfs, ignore_index=True)
    
    # Add cohort-specific fixed effects identifier
    stacked['cohort_country'] = stacked['cohort_id'] + '_' + stacked['iso3c']
    
    return stacked

## 5. Build Stacked Datasets

In [6]:
# Build stacked dataset for positive reforms
print("Building stacked dataset for positive reforms...")
stacked_pos = build_stacked_dataset(
    df, events_pos,
    K=K, L=L,
    control_type=EVENT_CONFIG['control_type']
)

print(f"Positive reforms stacked:")
print(f"  Total observations: {len(stacked_pos)}")
print(f"  Cohorts: {stacked_pos['cohort_id'].nunique()}")
print(f"  Countries: {stacked_pos['iso3c'].nunique()}")
print(f"  Event times: {sorted(stacked_pos['event_time'].unique())}")

Building stacked dataset for positive reforms...
Positive reforms stacked:
  Total observations: 45170
  Cohorts: 48
  Countries: 165
  Event times: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


In [7]:
# Build stacked dataset for negative reforms
print("\nBuilding stacked dataset for negative reforms...")
stacked_neg = build_stacked_dataset(
    df, events_neg,
    K=K, L=L,
    control_type=EVENT_CONFIG['control_type']
)

print(f"Negative reforms stacked:")
print(f"  Total observations: {len(stacked_neg)}")
print(f"  Cohorts: {stacked_neg['cohort_id'].nunique()}")
print(f"  Countries: {stacked_neg['iso3c'].nunique()}")
print(f"  Event times: {sorted(stacked_neg['event_time'].unique())}")


Building stacked dataset for negative reforms...
Negative reforms stacked:
  Total observations: 18370
  Cohorts: 20
  Countries: 165
  Event times: [np.int64(-2), np.int64(-1), np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]


## 6. Create Event-Time Dummies

In [8]:
def add_event_time_dummies(df, K=2, L=4, omit_period=-1):
    """
    Add event-time dummy variables.
    
    Parameters
    ----------
    df : DataFrame with 'event_time' column
    K : int
        Number of pre-periods
    L : int
        Number of post-periods
    omit_period : int
        Reference period to omit (default: -1)
        
    Returns
    -------
    DataFrame with event-time dummies
    """
    df = df.copy()
    
    for k in range(-K, L+1):
        if k == omit_period:
            continue  # Reference period
        col_name = f'et_{k}' if k < 0 else f'et_p{k}'  # et_-2, et_-1, et_p0, et_p1, ...
        df[col_name] = (df['event_time'] == k).astype(int)
    
    # Create interaction dummies (treated × event_time)
    for k in range(-K, L+1):
        if k == omit_period:
            continue
        dummy_col = f'et_{k}' if k < 0 else f'et_p{k}'
        interact_col = f'treat_et_{k}' if k < 0 else f'treat_et_p{k}'
        df[interact_col] = df['treated'] * df[dummy_col]
    
    return df

# Add event-time dummies
stacked_pos = add_event_time_dummies(stacked_pos, K=K, L=L)
stacked_neg = add_event_time_dummies(stacked_neg, K=K, L=L)

# Check interaction columns
interact_cols = [c for c in stacked_pos.columns if c.startswith('treat_et_')]
print(f"\nTreatment × Event-time interactions ({len(interact_cols)} columns):")
for col in sorted(interact_cols):
    print(f"  {col}: {stacked_pos[col].sum()} treated obs")


Treatment × Event-time interactions (6 columns):
  treat_et_-2: 48 treated obs
  treat_et_p0: 48 treated obs
  treat_et_p1: 48 treated obs
  treat_et_p2: 48 treated obs
  treat_et_p3: 47 treated obs
  treat_et_p4: 39 treated obs


## 7. Prepare Outcome Variables (Forward Growth)

In [9]:
def prepare_outcomes(df):
    """
    Prepare outcome variables for LP estimation.
    
    For LP-DiD, we use growth forward from the base period.
    Y_{i,t+h} - Y_{i,t-1} or equivalently sum of growth from t to t+h
    """
    df = df.copy()
    
    # Check outcome availability
    if 'growth' in df.columns:
        # Forward growth rates are already in the data
        # For event-study, we want cumulative growth from t-1 to t+h
        # This requires recomputing based on ln_gdp_pc
        pass
    
    if 'ln_gdp_pc' in df.columns:
        # Compute cumulative log difference from base period (t-1)
        # For each cohort, t-1 is event_time = -1
        
        # Get base period ln_gdp_pc for each unit in each cohort
        base_period = df[df['event_time'] == -1][['cohort_id', 'iso3c', 'ln_gdp_pc']].copy()
        base_period = base_period.rename(columns={'ln_gdp_pc': 'ln_gdp_pc_base'})
        
        df = df.merge(base_period, on=['cohort_id', 'iso3c'], how='left')
        
        # Cumulative outcome: Y_{t+h} - Y_{t-1}
        df['y_cumulative'] = df['ln_gdp_pc'] - df['ln_gdp_pc_base']
    
    return df

stacked_pos = prepare_outcomes(stacked_pos)
stacked_neg = prepare_outcomes(stacked_neg)

print("Outcome variable summary (stacked_pos):")
if 'y_cumulative' in stacked_pos.columns:
    print(stacked_pos.groupby('event_time')['y_cumulative'].describe().round(3))

Outcome variable summary (stacked_pos):
             count   mean    std    min    25%    50%    75%    max
event_time                                                         
-2          5650.0 -0.044  0.221 -1.251 -0.151 -0.069  0.055  1.250
-1          6023.0  0.000  0.000  0.000  0.000  0.000  0.000  0.000
 0          6023.0  0.049  0.237 -1.250 -0.046  0.077  0.158  1.251
 1          6023.0  0.133  0.339 -2.314 -0.032  0.160  0.289  1.363
 2          6022.0  0.239  0.388 -2.559  0.032  0.239  0.430  1.626
 3          5898.0  0.328  0.444 -2.598  0.096  0.320  0.541  1.838
 4          4954.0  0.411  0.498 -1.644  0.123  0.393  0.673  2.187


## 8. Validation Checks

In [10]:
def validate_stacked_data(df, name):
    """Validate stacked dataset integrity."""
    print(f"\nValidation: {name}")
    print("="*50)
    
    # 1. Check for duplicate: each cohort should have unique country-year combinations
    # Note: Same country CAN appear in multiple cohorts (expected in stacked design)
    dup_within_cohort = df.groupby(['cohort_id']).apply(
        lambda x: x.duplicated(subset=['iso3c', 'year']).any()
    )
    n_cohorts_with_dups = dup_within_cohort.sum()
    if n_cohorts_with_dups > 0:
        print(f"⚠ Warning: {n_cohorts_with_dups} cohorts have duplicate country-year")
    else:
        print("✓ No duplicate observations within cohorts")
    
    # 2. Check event_time range
    et_range = df['event_time'].unique()
    print(f"✓ Event-time range: {min(et_range)} to {max(et_range)}")
    
    # 3. Check treatment timing
    treated_at_0 = df[(df['treated'] == 1) & (df['event_time'] == 0)]
    n_cohorts = df['cohort_id'].nunique()
    print(f"✓ Cohorts: {n_cohorts}, treated at t=0: {len(treated_at_0)}")
    
    # 4. Check controls
    n_treated_units = df[df['treated'] == 1]['iso3c'].nunique()
    n_control_units = df[df['treated'] == 0]['iso3c'].nunique()
    print(f"✓ Treated units: {n_treated_units}, Control units: {n_control_units}")
    
    # 5. Check outcome availability
    if 'y_cumulative' in df.columns:
        n_outcome = df['y_cumulative'].notna().sum()
        pct = 100 * n_outcome / len(df)
        print(f"✓ Outcome available: {n_outcome} obs ({pct:.1f}%)")
    
    return True

validate_stacked_data(stacked_pos, "Positive Reforms")
validate_stacked_data(stacked_neg, "Negative Reforms")


Validation: Positive Reforms
⚠ Warning: 1 cohorts have duplicate country-year
✓ Event-time range: -2 to 4
✓ Cohorts: 48, treated at t=0: 49
✓ Treated units: 47, Control units: 164
✓ Outcome available: 40593 obs (89.8%)

Validation: Negative Reforms
⚠ Warning: 3 cohorts have duplicate country-year
✓ Event-time range: -2 to 4
✓ Cohorts: 20, treated at t=0: 23
✓ Treated units: 17, Control units: 160
✓ Outcome available: 15310 obs (83.2%)


True

## 9. Create Combined Dataset with Sign Indicator

In [11]:
# Add sign indicator
stacked_pos['reform_sign'] = 'positive'
stacked_neg['reform_sign'] = 'negative'

# Combine
stacked_combined = pd.concat([stacked_pos, stacked_neg], ignore_index=True)

print(f"\nCombined stacked dataset:")
print(f"  Total observations: {len(stacked_combined)}")
print(f"  Positive cohorts: {stacked_combined[stacked_combined['reform_sign'] == 'positive']['cohort_id'].nunique()}")
print(f"  Negative cohorts: {stacked_combined[stacked_combined['reform_sign'] == 'negative']['cohort_id'].nunique()}")


Combined stacked dataset:
  Total observations: 63592
  Positive cohorts: 48
  Negative cohorts: 20


## 10. Summary Statistics by Event Time

In [12]:
print("\nSummary by event time (Positive Reforms):")
summary_pos = stacked_pos.groupby('event_time').agg({
    'treated': 'sum',
    'y_cumulative': ['count', 'mean', 'std'],
    'cohort_id': 'nunique'
}).round(3)
summary_pos.columns = ['n_treated', 'n_outcome', 'mean_y', 'std_y', 'n_cohorts']
print(summary_pos)

print("\nSummary by event time (Negative Reforms):")
summary_neg = stacked_neg.groupby('event_time').agg({
    'treated': 'sum',
    'y_cumulative': ['count', 'mean', 'std'],
    'cohort_id': 'nunique'
}).round(3)
summary_neg.columns = ['n_treated', 'n_outcome', 'mean_y', 'std_y', 'n_cohorts']
print(summary_neg)


Summary by event time (Positive Reforms):
            n_treated  n_outcome  mean_y  std_y  n_cohorts
event_time                                                
-2                 49       5650  -0.044  0.221         48
-1                 49       6023   0.000  0.000         48
 0                 49       6023   0.049  0.237         48
 1                 49       6023   0.133  0.339         48
 2                 49       6022   0.239  0.388         48
 3                 48       5898   0.328  0.444         47
 4                 40       4954   0.411  0.498         39

Summary by event time (Negative Reforms):
            n_treated  n_outcome  mean_y  std_y  n_cohorts
event_time                                                
-2                 14       1788  -0.100  0.185         13
-1                 23       2652   0.000  0.000         20
 0                 23       2650   0.103  0.161         20
 1                 21       2356   0.194  0.270         18
 2                 21       2

## 11. Save Outputs

In [13]:
# Save stacked datasets
output_pos = DATA_CLEAN / 'stacked_events_pos.parquet'
output_neg = DATA_CLEAN / 'stacked_events_neg.parquet'
output_combined = DATA_CLEAN / 'stacked_events_combined.parquet'

stacked_pos.to_parquet(output_pos, index=False)
stacked_neg.to_parquet(output_neg, index=False)
stacked_combined.to_parquet(output_combined, index=False)

print(f"\n✓ Saved positive reforms to {output_pos}")
print(f"✓ Saved negative reforms to {output_neg}")
print(f"✓ Saved combined dataset to {output_combined}")

# Save metadata
meta = {
    'config': EVENT_CONFIG,
    'timestamp': datetime.now().isoformat(),
    'positive_reforms': {
        'n_obs': len(stacked_pos),
        'n_cohorts': int(stacked_pos['cohort_id'].nunique()),
        'n_treated': int(stacked_pos[stacked_pos['treated'] == 1]['iso3c'].nunique()),
        'n_control': int(stacked_pos[stacked_pos['treated'] == 0]['iso3c'].nunique()),
    },
    'negative_reforms': {
        'n_obs': len(stacked_neg),
        'n_cohorts': int(stacked_neg['cohort_id'].nunique()),
        'n_treated': int(stacked_neg[stacked_neg['treated'] == 1]['iso3c'].nunique()),
        'n_control': int(stacked_neg[stacked_neg['treated'] == 0]['iso3c'].nunique()),
    },
    'variables': {
        'outcome': 'y_cumulative (ln GDP per capita relative to t-1)',
        'treatment': 'treated × event_time interactions',
        'fixed_effects': 'cohort_id × iso3c (cohort-country FE)',
    }
}

meta_path = OUTPUT_LOGS / 'stacked_events_metadata.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print(f"✓ Saved metadata to {meta_path}")


✓ Saved positive reforms to /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/03_clean/stacked_events_pos.parquet
✓ Saved negative reforms to /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/03_clean/stacked_events_neg.parquet
✓ Saved combined dataset to /Users/gabrielsaco/Documents/GitHub/economic-freedom/data/03_clean/stacked_events_combined.parquet
✓ Saved metadata to /Users/gabrielsaco/Documents/GitHub/economic-freedom/output/logs/stacked_events_metadata.json


## 12. Final Summary

In [14]:
print("\n" + "="*60)
print("05_BUILD_STACKED_EVENT_DATA COMPLETE")
print("="*60)
print(f"Event window: t={-K} to t=+{L} ({window_size} periods)")
print(f"Control type: {EVENT_CONFIG['control_type']}")
print(f"Positive reform cohorts: {stacked_pos['cohort_id'].nunique()}")
print(f"Negative reform cohorts: {stacked_neg['cohort_id'].nunique()}")
print(f"Total stacked observations: {len(stacked_combined)}")
print("="*60)


05_BUILD_STACKED_EVENT_DATA COMPLETE
Event window: t=-2 to t=+4 (7 periods)
Control type: not_yet_treated
Positive reform cohorts: 48
Negative reform cohorts: 20
Total stacked observations: 63592
